# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.02 · Alternativa local ligera y diagnóstico

Mantiene una ruta sin costo de API con Qwen3-1.7B y el mismo prompt compacto; es un fallback, no se mezcla con la campaña principal Flash→Pro.

Qwen3 es una familia multilingüe de pesos abiertos [1]. La revisión exacta de `Qwen/Qwen3-1.7B` se fija mediante su tarjeta oficial [2]. Esta ruta reduce memoria respecto de 4B, pero su calidad debe calibrarse por separado y sus propuestas no son *ground truth* [3].

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Proveedor local ligero

In [ ]:
from moderacion_peru.providers import HuggingFaceProvider
from moderacion_peru.io import read_jsonl
provider=HuggingFaceProvider(model='Qwen/Qwen3-1.7B',revision='70d244cc86ccca08cf5af4e1e306ecf908b1ad5e',device='auto',records_per_request=5,inference_batch_size=4,max_new_tokens=256)
SOURCE=ROOT/'datos/processed/chunks_v2.jsonl'
OUTPUT=ROOT/'datos/etiquetado/fallback_hf/qwen3_1_7b_v2.jsonl'
ERRORS=OUTPUT.with_suffix('.errors.jsonl')
show_result('Estado del fallback',provider.probe(),tone='neutral')

## Ejecución por lotes y avance

In [ ]:
from tqdm.auto import tqdm
from moderacion_peru.labeling import annotate_batched_incremental
RUN_FALLBACK=False
LIMIT=20  # Use None para todos los pendientes después del smoke test.
local_progress={'bar':None}
def report_local_progress(event):
    if event['status']=='started':
        local_progress['bar']=tqdm(total=event['selected'],desc='Fallback Qwen3-1.7B',unit='chunk'); return
    bar=local_progress.get('bar')
    if bar is not None and event.get('advance'):
        bar.update(event['advance']); bar.set_postfix(ok=event['labeled'],errores=event['errors'])
    if event['status']=='finished' and bar is not None: bar.close(); local_progress['bar']=None
if RUN_FALLBACK:
    fallback_result=annotate_batched_incremental(read_jsonl(SOURCE),provider,OUTPUT,error_path=ERRORS,limit=LIMIT,processing_batch_size=20,progress_callback=report_local_progress,run_metadata={'provider':provider.probe(),'role':'independent_fallback'})
    show_result('Resultado local',fallback_result,tone='success')
else:
    show_callout('Fallback desactivado','La campaña principal se ejecuta completa en 02_01. Active esta ruta solo para un diagnóstico independiente.',tone='neutral')

## Referencias

[1] A. Yang, A. Li, B. Yang, et al., "Qwen3 Technical Report," arXiv:2505.09388, 2025, doi: 10.48550/arXiv.2505.09388.

[2] Qwen Team, "Model Card: Qwen/Qwen3-1.7B," Hugging Face Hub, revision 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-1.7B/tree/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e. Accessed: Aug. 7, 2026.

[3] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.